In [0]:
# ============================================
# CAMADA GOLD - Star Schema + OBT para DS
# ============================================
from pyspark.sql.functions import (
    current_timestamp, col, count, sum, avg, max,
    when, round, countDistinct, hour, dayofweek
)

BUCKET_PATH = "s3://data-lake-portfolio-priscila-2026"
SILVER_PATH = f"{BUCKET_PATH}/silver"
GOLD_PATH   = f"{BUCKET_PATH}/gold"

# Lendo Silver
clientes   = spark.read.format("delta").load(f"{SILVER_PATH}/clientes")
transacoes = spark.read.format("delta").load(f"{SILVER_PATH}/transacoes")
logs       = spark.read.format("delta").load(f"{SILVER_PATH}/logs_seguranca")

# ── STAR SCHEMA ──────────────────────────────

# Dimensão Cliente
dim_cliente = clientes.select(
    "cliente_id", "nome", "cidade", "data_nascimento"
).withColumn("_gold_at", current_timestamp())

dim_cliente.write.format("delta").mode("overwrite").save(f"{GOLD_PATH}/dim_cliente")
print(f"✅ dim_cliente: {dim_cliente.count()} linhas")

# Fato Transações (métricas financeiras por cliente)
fato_transacoes = transacoes \
    .groupBy("cliente_id") \
    .agg(
        count("transacao_id").alias("total_transacoes"),
        sum(when(col("status") == "Aprovada",  col("valor"))).alias("valor_total_aprovado"),
        sum(when(col("status") == "Negada",    col("valor"))).alias("valor_total_negado"),
        count(when(col("status") == "Aprovada", 1)).alias("qtd_aprovadas"),
        count(when(col("status") == "Negada",   1)).alias("qtd_negadas"),
        avg("valor").alias("ticket_medio"),
        max("data_hora").alias("ultima_transacao")
    ) \
    .withColumn("taxa_aprovacao", 
        round(col("qtd_aprovadas") / col("total_transacoes") * 100, 2)) \
    .withColumn("_gold_at", current_timestamp())

fato_transacoes.write.format("delta").mode("overwrite").save(f"{GOLD_PATH}/fato_transacoes")
print(f"✅ fato_transacoes: {fato_transacoes.count()} linhas")

# ── OBT - One Big Table para Data Science ────
# Features de comportamento + risco para modelo de ML

features_risco = logs \
    .groupBy("cliente_id") \
    .agg(
        count("log_id").alias("total_logins"),
        count(when(col("sucesso_login") == False, 1)).alias("logins_falhos"),
        avg("score_risco_dispositivo").alias("score_risco_medio"),
        max("score_risco_dispositivo").alias("score_risco_maximo"),
        countDistinct("dispositivo").alias("qtd_dispositivos_distintos"),
        count(when(col("dispositivo") == "iOS",     1)).alias("logins_ios"),
        count(when(col("dispositivo") == "Android", 1)).alias("logins_android"),
        count(when(col("dispositivo") == "Web",     1)).alias("logins_web")
    ) \
    .withColumn("taxa_falha_login",
        round(col("logins_falhos") / col("total_logins") * 100, 2))

obt_fraude = dim_cliente.drop("_gold_at") \
    .join(fato_transacoes.drop("_gold_at"), "cliente_id", "left") \
    .join(features_risco, "cliente_id", "left") \
    .withColumn("flag_alto_risco", 
        when(
            (col("taxa_falha_login") > 20) | 
            (col("score_risco_medio") > 70) |
            (col("taxa_aprovacao") < 50), True
        ).otherwise(False)
    ) \
    .withColumn("_gold_at", current_timestamp())

obt_fraude.write.format("delta").mode("overwrite").save(f"{GOLD_PATH}/obt_fraude_features")
print(f"✅ obt_fraude_features: {obt_fraude.count()} linhas")

# Preview dos clientes de alto risco
print("\n🚨 Amostra de clientes flag_alto_risco=True:")
obt_fraude.filter(col("flag_alto_risco") == True) \
    .select("nome", "cidade", "taxa_aprovacao", "taxa_falha_login", "score_risco_medio") \
    .show(5, truncate=False)

print("\n🥇 Camada Gold concluída!")